In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

for df in (df_metric, df_trace, df_log, df_error):
    # parse timestamp column (unix seconds) to UTC datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to aggregate numeric-series groups with required stats
def aggregate_time_series(df, group_cols, value_col='value', top_n=50):
    # Keep full aggregated table in memory for later use
    agg = df.groupby(group_cols).agg(
        count=(value_col, 'count'),
        min_ts=('timestamp', 'min'),
        max_ts=('timestamp', 'max'),
        p95=(value_col, lambda s: s.quantile(0.95)),
        p5=(value_col, lambda s: s.quantile(0.05)),
    ).reset_index()
    # Sort by count desc and return top_n for compact display
    agg_sorted = agg.sort_values('count', ascending=False).head(top_n)
    return agg, agg_sorted

# Aggregate metric.csv by (cmdb_id, kpi_name)
metric_agg_full, metric_agg_top = aggregate_time_series(df_metric, ['cmdb_id', 'kpi_name'], 'value', top_n=50)

# Aggregate trace.csv by (cmdb_id, trace_name)
trace_agg_full, trace_agg_top = aggregate_time_series(df_trace, ['cmdb_id', 'trace_name'], 'value', top_n=50)

# Aggregate log.csv by (cmdb_id, log_name)
log_agg_full, log_agg_top = aggregate_time_series(df_log, ['cmdb_id', 'log_name'], 'value', top_n=50)

# Unique cmdb_id values across all files (as a compact pandas Series)
unique_cmdb_ids = pd.Series(
    sorted(set(df_metric['cmdb_id'].dropna().unique())
           | set(df_trace['cmdb_id'].dropna().unique())
           | set(df_log['cmdb_id'].dropna().unique())
           | set(df_error['cmdb_id'].dropna().unique())),
    name='cmdb_id'
)

# Earliest and latest timestamp present in each file
file_bounds = pd.DataFrame([
    {
        'file': 'metric.csv',
        'earliest_ts': df_metric['timestamp'].min(),
        'latest_ts': df_metric['timestamp'].max()
    },
    {
        'file': 'trace.csv',
        'earliest_ts': df_trace['timestamp'].min(),
        'latest_ts': df_trace['timestamp'].max()
    },
    {
        'file': 'log.csv',
        'earliest_ts': df_log['timestamp'].min(),
        'latest_ts': df_log['timestamp'].max()
    },
    {
        'file': 'error_logs.csv',
        'earliest_ts': df_error['timestamp'].min(),
        'latest_ts': df_error['timestamp'].max()
    },
])

# Keep full aggregated tables in memory as metric_agg_full, trace_agg_full, log_agg_full
# Display compact summaries (top results)
metric_agg_top, trace_agg_top, log_agg_top, unique_cmdb_ids, file_bounds

```
Out[1]:
```
```python
# Build a concise plain-English summary string based on previous aggregations (variables are in kernel)
n_cmdb = int(unique_cmdb_ids.shape[0])
metric_start = file_bounds.loc[file_bounds['file']=='metric.csv', 'earliest_ts'].iat[0]
metric_end = file_bounds.loc[file_bounds['file']=='metric.csv', 'latest_ts'].iat[0]
trace_start = file_bounds.loc[file_bounds['file']=='trace.csv', 'earliest_ts'].iat[0]
trace_end = file_bounds.loc[file_bounds['file']=='trace.csv', 'latest_ts'].iat[0]
log_start = file_bounds.loc[file_bounds['file']=='log.csv', 'earliest_ts'].iat[0]
log_end = file_bounds.loc[file_bounds['file']=='log.csv', 'latest_ts'].iat[0]

# Check error_logs presence
error_empty = df_error.empty or df_error['timestamp'].isna().all()

# Extract top example rows for compact highlights
m0 = metric_agg_top.iloc[0] if len(metric_agg_top) > 0 else None
t0 = trace_agg_top.iloc[0] if len(trace_agg_top) > 0 else None
l0 = log_agg_top.iloc[0] if len(log_agg_top) > 0 else None

summary = (
    "Summary of loaded telemetry (plain English):\n"
    f"- Files with timestamped data: metric.csv, trace.csv, log.csv. Each covers {metric_start.isoformat()} to {metric_end.isoformat()} (UTC) for metric and similar for trace/log:\n"
    f"  metric: {metric_start.isoformat()} → {metric_end.isoformat()} (UTC)\n"
    f"  trace:  {trace_start.isoformat()} → {trace_end.isoformat()} (UTC)\n"
    f"  log:    {log_start.isoformat()} → {log_end.isoformat()} (UTC)\n"
    f"- error_logs.csv contains no valid timestamps (no timestamped error log entries found).\n"
    f"- Number of unique cmdb_id values across all files: {n_cmdb}.\n\n"
    "Compact aggregation highlights (top examples):\n"
    f"- Top metric group: {m0['cmdb_id']} / {m0['kpi_name']} — count={int(m0['count'])}, p95={m0['p95']}, p5={m0['p5']}.\n"
    f"- Top trace group:  {t0['cmdb_id']} / {t0['trace_name']} — count={int(t0['count'])}, p95={t0['p95']}, p5={t0['p5']}.\n"
    f"- Top log group:    {l0['cmdb_id']} / {l0['log_name']} — count={int(l0['count'])}, p95={l0['p95']}, p5={l0['p5']}.\n\n"
    "Notes: Aggregations were computed per (cmdb_id, kpi/trace/log name) and include count, min/max timestamp, P95 and P5 of values. Many groups show count=25 (full minute series from 18:31–18:55 UTC)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count                    min_ts                    max_ts           p95            p5
358           ts-voucher-service      socket     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  7.888974e-01  3.816275e-01
1    ts-admin-basic-info-service      diskio     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  5.759602e+04  4.500757e+04
2    ts-admin-basic-info-service       error     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  1.796847e-02  1.750000e-02
..                           ...         ...    ...                       ...                       ...           ...           ...
70             ts-config-service      diskio     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  5.867538e+04  4.210274e+04
55              ts-basic-service      diskio     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  5.796645e+04  4.418580e+04
56              ts-basic-service  latency-50     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  8.493603e-03  8.271711e-03
57              ts-basic-service  latency-90     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  3.558435e-02  3.245025e-02
58              ts-basic-service         mem     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  2.491954e+08  2.486075e+08

[50 rows x 7 columns],                          cmdb_id                                         trace_name  count                    min_ts                    max_ts        p95         p5
775              ts-user-service                 trace.to_ts-user-service.row_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  22.000000   6.800000
0                           root  trace.to_ts-admin-basic-info-service.duration_...     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.014812   0.011099
1                           root  trace.to_ts-admin-basic-info-service.duration_p95     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.020884   0.012086
2                           root    trace.to_ts-admin-basic-info-service.error_rate     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.000000   0.000000
3                           root     trace.to_ts-admin-basic-info-service.row_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  20.800000  13.400000
..                           ...                                                ...    ...                       ...                       ...        ...        ...
57                          root           trace.to_ts-travel2-service.duration_p95     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.401208   0.324960
58                          root             trace.to_ts-travel2-service.error_rate     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.000000   0.000000
59                          root              trace.to_ts-travel2-service.row_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  91.600000  61.000000
60   ts-admin-basic-info-service                      trace.from_root.duration_mean     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.014812   0.011099
61   ts-admin-basic-info-service                       trace.from_root.duration_p95     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.020884   0.012086

[50 rows x 7 columns],                         cmdb_id         log_name  count                    min_ts                    max_ts   p95    p5
0   ts-admin-basic-info-service  log.error_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.0   0.0
1   ts-admin-basic-info-service    log.row_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  31.6  19.8
2       ts-admin-travel-service  log.error_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   0.0   0.0
3       ts-admin-travel-service    log.row_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  32.4  21.6
7          ts-assurance-service    log.row_count     25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  17.0   4.4
..                          ...              ...    ...                       ...                       ...   ...   ...
18       ts-consign-price-mongo  log.error_count     24 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00   0.0   0.0
14              ts-config-mongo  log.error_count     24 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00   0.0   0.0
15              ts-config-mongo    log.row_count     24 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00   1.0   1.0
16             ts-consign-mongo  log.error_count     24 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00   0.0   0.0
17             ts-consign-mongo    log.row_count     24 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00   1.0   1.0

[50 rows x 7 columns], 0                             root
1      ts-admin-basic-info-service
2           ts-admin-order-service
3           ts-admin-route-service
4          ts-admin-travel-service
                  ...             
64                   ts-user-mongo
65                 ts-user-service
66    ts-verification-code-service
67                ts-voucher-mysql
68              ts-voucher-service
Name: cmdb_id, Length: 69, dtype: object,              file               earliest_ts                 latest_ts
0      metric.csv 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00
1       trace.csv 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00
2         log.csv 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00
3  error_logs.csv                       NaT                       NaT)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global thresholds (P95 and P5) for each (cmdb_id, kpi_name) using full metric series
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_p95=lambda s: s.quantile(0.95), global_p5=lambda s: s.quantile(0.05))
    .reset_index()
)

# 2) Filter metric rows to the incident window [2024-02-20 18:28:18 UTC, 2024-02-20 18:58:18 UTC]
start_ts = pd.Timestamp("2024-02-20 18:28:18", tz='UTC')
end_ts = pd.Timestamp("2024-02-20 18:58:18", tz='UTC')
metric_filtered_full = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Join global thresholds into filtered data and mark anomalies
metric_filtered_full = metric_filtered_full.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Anomaly condition: value >= global_p95 OR value <= global_p5
# Comparisons with NaN will yield False (not marked as anomaly)
metric_filtered_full['is_anomaly'] = (
    (metric_filtered_full['value'] >= metric_filtered_full['global_p95']) |
    (metric_filtered_full['value'] <= metric_filtered_full['global_p5'])
)

# 4) Aggregate anomaly results per (cmdb_id, kpi_name)
metric_anomalies_full = metric_filtered_full[metric_filtered_full['is_anomaly']].copy()

anomaly_summary = (
    metric_anomalies_full
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        earliest_anomaly_ts=('timestamp', 'min'),
        latest_anomaly_ts=('timestamp', 'max'),
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        global_p95=('global_p95', 'first'),
        global_p5=('global_p5', 'first'),
    )
)

# 5A) Compact anomaly summary top 20 rows sorted by anomaly_count desc
anomaly_summary_top20 = anomaly_summary.sort_values('anomaly_count', ascending=False).head(20)

# 5B) Anomaly detail rows limited to top 100 sorted by timestamp ascending (select compact columns)
anomaly_detail_top100 = (
    metric_anomalies_full
    .sort_values('timestamp', ascending=True)
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'global_p95', 'global_p5']]
    .head(100)
)

# Keep full filtered metric table and full anomaly table in kernel variables:
# metric_filtered_full, metric_anomalies_full are stored above.

# Display compact outputs
anomaly_summary_top20, anomaly_detail_top100

```
Out[2]:
```
```python
# Build a concise plain-English summary of the anomaly analysis using existing kernel variables
total_anomaly_points = int(metric_anomalies_full.shape[0])
affected_groups = int(anomaly_summary.shape[0])
earliest_anom = metric_anomalies_full['timestamp'].min()
latest_anom = metric_anomalies_full['timestamp'].max()

top5 = (
    anomaly_summary
    .sort_values('anomaly_count', ascending=False)
    .head(5)
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count']]
)
top5_list = [f"{r.cmdb_id} / {r.kpi_name} (count={int(r.anomaly_count)})" for _, r in top5.iterrows()]

summary = (
    "Anomaly analysis summary (plain English):\n"
    f"- Total anomaly data points detected in the incident window: {total_anomaly_points}.\n"
    f"- Number of affected (cmdb_id, kpi_name) groups: {affected_groups}.\n"
    f"- Earliest anomaly timestamp: {earliest_anom.isoformat()} (UTC).\n"
    f"- Latest anomaly timestamp:   {latest_anom.isoformat()} (UTC).\n"
    "- Top affected groups (by anomaly count):\n"
    + "\n".join([f"  {i+1}. {s}" for i, s in enumerate(top5_list)]) + "\n\n"
    "Notes:\n"
    "- Global P95/P5 thresholds were computed using the full metric series before filtering.\n"
    "- Rows within the incident window [2024-02-20 18:28:18 UTC, 2024-02-20 18:58:18 UTC] were marked anomalous when value >= global P95 or value <= global P5.\n"
    "- Full filtered metric table is stored in variable: metric_filtered_full\n"
    "- Full anomaly detail table is stored in variable: metric_anomalies_full\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts     max_value    min_value    global_p95    global_p5
101         ts-contacts-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  7.000000e+00          7.0  7.000000e+00          7.0
278    ts-ticket-office-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  2.000000e+00          2.0  2.000000e+00          2.0
282  ts-ticket-office-service   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  3.000000e+00          1.0  1.506667e+00          1.0
320          ts-travel2-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  8.000000e+00          8.0  8.000000e+00          8.0
142   ts-inside-payment-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  4.000000e+00          4.0  4.000000e+00          4.0
83     ts-consign-price-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  5.000000e+00          5.0  5.000000e+00          5.0
234            ts-route-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  1.800000e+01         18.0  1.800000e+01         18.0
79           ts-consign-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  5.000000e+00          5.0  5.000000e+00          5.0
293            ts-train-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  1.900000e+01         19.0  1.900000e+01         19.0
152           ts-news-service   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  1.000000e+00          1.0  1.000000e+00          1.0
356        ts-voucher-service   diskio             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  0.000000e+00          0.0  0.000000e+00          0.0
18    ts-admin-travel-service    error             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  0.000000e+00          0.0  0.000000e+00          0.0
329           ts-ui-dashboard   diskio             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  0.000000e+00          0.0  0.000000e+00          0.0
354          ts-voucher-mysql   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  2.000000e+00          2.0  2.000000e+00          2.0
353          ts-voucher-mysql      mem             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  6.004982e+08  600498176.0  6.004982e+08  600498176.0
352          ts-voucher-mysql   diskio             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  0.000000e+00          0.0  0.000000e+00          0.0
330           ts-ui-dashboard    error             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  6.675000e-02          0.0  0.000000e+00          0.0
168      ts-order-other-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  1.800000e+01         18.0  1.800000e+01         18.0
219            ts-price-mongo   socket             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  1.900000e+01         19.0  1.900000e+01         19.0
51          ts-avatar-service   diskio             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  0.000000e+00          0.0  0.000000e+00          0.0,                     timestamp                      cmdb_id    kpi_name         value    global_p95     global_p5
0   2024-02-20 18:31:00+00:00  ts-admin-basic-info-service         cpu  1.053346e+00  7.888974e-01  3.816275e-01
213 2024-02-20 18:31:00+00:00          ts-preserve-service         mem  2.240746e+08  2.274397e+08  2.241299e+08
218 2024-02-20 18:31:00+00:00               ts-price-mongo         mem  6.823507e+07  6.851417e+07  6.824406e+07
219 2024-02-20 18:31:00+00:00               ts-price-mongo      socket  1.900000e+01  1.900000e+01  1.900000e+01
224 2024-02-20 18:31:00+00:00             ts-price-service         mem  2.548882e+08  2.587243e+08  2.549889e+08
..                        ...                          ...         ...           ...           ...           ...
58  2024-02-20 18:31:00+00:00             ts-basic-service         mem  2.485384e+08  2.491954e+08  2.486075e+08
71  2024-02-20 18:31:00+00:00            ts-config-service  latency-50  5.413803e-03  4.239445e-03  3.531832e-03
31  2024-02-20 18:31:00+00:00           ts-assurance-mongo      socket  4.000000e+00  5.000000e+00  4.000000e+00
27  2024-02-20 18:31:00+00:00        ts-admin-user-service      socket  3.000000e+00  3.420000e+00  3.000000e+00
2   2024-02-20 18:31:00+00:00  ts-admin-basic-info-service       error  0.000000e+00  0.000000e+00  0.000000e+00

[100 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse prior variables: anomaly_summary, metric_anomalies_full, metric_filtered_full
# Define candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Filter anomaly_summary to only the candidate components and select requested columns
candidate_anomaly_summary = (
    anomaly_summary[anomaly_summary['cmdb_id'].isin(candidates)]
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'latest_anomaly_ts',
             'max_value', 'min_value', 'global_p95', 'global_p5']]
    .sort_values('anomaly_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 2) Compute aggregates for these candidates
# (A) total anomaly points across all their KPIs (count rows in metric_anomalies_full for these cmdb_ids)
candidate_anomaly_points = metric_anomalies_full[metric_anomalies_full['cmdb_id'].isin(candidates)].shape[0]

# (B) number of distinct (cmdb_id, kpi_name) groups affected
candidate_distinct_groups = (
    metric_anomalies_full[metric_anomalies_full['cmdb_id'].isin(candidates)]
    .drop_duplicates(subset=['cmdb_id', 'kpi_name'])
    .shape[0]
)

# Pack aggregates into a small DataFrame for compact display
candidate_aggregates = pd.DataFrame([{
    'total_anomaly_points': int(candidate_anomaly_points),
    'distinct_affected_groups': int(candidate_distinct_groups)
}])

# 3) Sample anomaly detail rows for these candidates (top 50 by timestamp ascending)
candidate_anomaly_detail = (
    metric_anomalies_full[metric_anomalies_full['cmdb_id'].isin(candidates)]
    .loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'global_p95', 'global_p5']]
    .sort_values('timestamp', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# Store results in kernel variables for next steps:
# candidate_anomaly_summary, candidate_aggregates, candidate_anomaly_detail

candidate_anomaly_summary, candidate_aggregates, candidate_anomaly_detail

```
Out[3]:
```
Summary for the five candidate services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):

1) Filtered anomaly summary (top results)
- Top affected KPI groups (by anomaly count):
  1. ts-travel-service / error — 6 anomaly points (earliest 2024-02-20T18:50:00Z, latest 18:55:00Z)
  2. ts-order-service / socket — 5 anomaly points (earliest 18:32:00Z, latest 18:53:00Z)
  3. ts-train-service / socket — 5 anomaly points (earliest 18:34:00Z, latest 18:55:00Z)
  4. ts-auth-service / cpu — 4 anomaly points (earliest 18:34:00Z, latest 18:50:00Z)
  5. ts-auth-service / latency-90 — 4 anomaly points (earliest 18:33:00Z, latest 18:55:00Z)
- Other repeated anomalies for ts-auth-service and ts-order-service include latency-50, diskio, mem, workload, and socket KPIs (many with 4 anomaly points).
- The candidate summary table includes: cmdb_id, kpi_name, anomaly_count, earliest_anomaly_ts, latest_anomaly_ts, max_value, min_value, global_p95, global_p5.

2) Aggregate counts for the five candidates
- Total anomaly points across all their KPIs (within the incident window): 148
- Number of distinct (cmdb_id, kpi_name) groups affected: 36

3) Sample anomaly detail rows (examples)
- A sample of anomaly rows (up to 50) sorted by timestamp shows anomalies starting at 2024-02-20T18:31:00Z and continuing (sample rows shown through ~18:39:00Z in the returned subset). The sample contains various KPI types: mem, cpu, latency-50/90, diskio, socket, workload, error, etc., with each row including timestamp, cmdb_id, kpi_name, value, global_p95, global_p5.

Stored variables for follow-up:
- candidate_anomaly_summary (filtered summary table)
- candidate_aggregates (total_anomaly_points, distinct_affected_groups)
- candidate_anomaly_detail (sample anomaly rows)

If you want, I can (a) display the full candidate_anomaly_summary table, (b) show the earliest/latest anomaly timestamps across only these candidates, or (c) filter the anomaly detail to a specific KPI or narrower time range.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts     max_value     min_value    global_p95     global_p5
0   ts-travel-service       error              6 2024-02-20 18:50:00+00:00 2024-02-20 18:55:00+00:00  2.392857e-02  0.000000e+00  1.794643e-02  0.000000e+00
1    ts-order-service      socket              5 2024-02-20 18:32:00+00:00 2024-02-20 18:53:00+00:00  1.478333e+01  1.205000e+01  1.464000e+01  1.211667e+01
2    ts-train-service      socket              5 2024-02-20 18:34:00+00:00 2024-02-20 18:55:00+00:00  2.500000e+01  2.375000e+01  2.500000e+01  2.380333e+01
3     ts-auth-service         cpu              4 2024-02-20 18:34:00+00:00 2024-02-20 18:50:00+00:00  1.942582e+01  1.419141e+01  1.885292e+01  1.475823e+01
4     ts-auth-service  latency-90              4 2024-02-20 18:33:00+00:00 2024-02-20 18:55:00+00:00  1.275052e+00  5.885402e-01  1.057945e+00  6.038476e-01
5     ts-auth-service  latency-50              4 2024-02-20 18:34:00+00:00 2024-02-20 18:55:00+00:00  3.339537e-01  2.176680e-01  2.980565e-01  2.268004e-01
6     ts-auth-service      diskio              4 2024-02-20 18:37:00+00:00 2024-02-20 18:55:00+00:00  6.962652e+04  3.993772e+04  5.873704e+04  4.562891e+04
7     ts-auth-service         mem              4 2024-02-20 18:31:00+00:00 2024-02-20 18:53:00+00:00  2.413324e+08  2.385778e+08  2.410114e+08  2.387147e+08
8    ts-order-service      diskio              4 2024-02-20 18:35:00+00:00 2024-02-20 18:55:00+00:00  7.065390e+04  3.314946e+04  6.248952e+04  4.443855e+04
9    ts-order-service  latency-50              4 2024-02-20 18:37:00+00:00 2024-02-20 18:54:00+00:00  9.526238e-03  7.154687e-03  9.086963e-03  7.213614e-03
10    ts-auth-service    workload              4 2024-02-20 18:38:00+00:00 2024-02-20 18:54:00+00:00  3.438967e+00  2.794667e+00  3.289333e+00  2.809727e+00
11    ts-auth-service      socket              4 2024-02-20 18:35:00+00:00 2024-02-20 18:53:00+00:00  1.500000e+01  1.246667e+01  1.475000e+01  1.249667e+01
12   ts-order-service         mem              4 2024-02-20 18:31:00+00:00 2024-02-20 18:54:00+00:00  2.609451e+08  2.570678e+08  2.608892e+08  2.576137e+08
13   ts-order-service    workload              4 2024-02-20 18:32:00+00:00 2024-02-20 18:49:00+00:00  9.319967e+00  4.509933e+00  9.077503e+00  4.680917e+00
14   ts-route-service         cpu              4 2024-02-20 18:32:00+00:00 2024-02-20 18:51:00+00:00  5.731560e+00  4.549364e+00  5.617819e+00  4.566993e+00
15   ts-route-service      diskio              4 2024-02-20 18:41:00+00:00 2024-02-20 18:55:00+00:00  6.424308e+04  3.444662e+04  6.267912e+04  4.051841e+04
16   ts-route-service  latency-50              4 2024-02-20 18:31:00+00:00 2024-02-20 18:48:00+00:00  4.016876e-03  3.478424e-03  3.880111e-03  3.501207e-03
17   ts-route-service  latency-90              4 2024-02-20 18:31:00+00:00 2024-02-20 18:50:00+00:00  8.719116e-03  7.615415e-03  8.607155e-03  7.823213e-03
18   ts-order-service  latency-90              4 2024-02-20 18:36:00+00:00 2024-02-20 18:43:00+00:00  8.403521e-02  1.794543e-02  4.562544e-02  1.868672e-02
19   ts-order-service         cpu              4 2024-02-20 18:31:00+00:00 2024-02-20 18:48:00+00:00  5.369633e+00  1.335544e+00  4.818914e+00  1.646401e+00,    total_anomaly_points  distinct_affected_groups
0                   148                        36,                    timestamp            cmdb_id    kpi_name         value    global_p95     global_p5
0  2024-02-20 18:31:00+00:00    ts-auth-service         mem  2.385778e+08  2.410114e+08  2.387147e+08
1  2024-02-20 18:31:00+00:00   ts-order-service         cpu  5.369633e+00  4.818914e+00  1.646401e+00
2  2024-02-20 18:31:00+00:00   ts-order-service         mem  2.570678e+08  2.608892e+08  2.576137e+08
3  2024-02-20 18:31:00+00:00   ts-route-service  latency-50  4.016876e-03  3.880111e-03  3.501207e-03
4  2024-02-20 18:31:00+00:00   ts-route-service  latency-90  8.719116e-03  8.607155e-03  7.823213e-03
..                       ...                ...         ...           ...           ...           ...
45 2024-02-20 18:38:00+00:00   ts-order-service      diskio  6.301784e+04  6.248952e+04  4.443855e+04
46 2024-02-20 18:38:00+00:00  ts-travel-service      socket  1.621667e+01  2.204000e+01  1.622667e+01
47 2024-02-20 18:39:00+00:00   ts-route-service    workload  4.066232e+01  4.063610e+01  3.467825e+01
48 2024-02-20 18:39:00+00:00   ts-train-service    workload  2.528760e+01  2.525142e+01  2.050225e+01
49 2024-02-20 18:39:00+00:00  ts-travel-service  latency-50  1.793382e-02  2.191251e-02  1.794468e-02

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing df_trace if present; otherwise load and parse timestamps as UTC
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) using full trace series
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(global_p95=lambda s: s.quantile(0.95), global_p5=lambda s: s.quantile(0.05))
    .reset_index()
)

# 2) Filter trace rows to the incident window [2024-02-20 18:28:18 UTC, 2024-02-20 18:58:18 UTC]
start_ts = pd.Timestamp("2024-02-20 18:28:18", tz='UTC')
end_ts   = pd.Timestamp("2024-02-20 18:58:18", tz='UTC')
trace_filtered = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Join global thresholds and mark anomalies: value >= global_p95 OR value <= global_p5
trace_filtered = trace_filtered.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
trace_filtered['is_anomaly'] = (
    (trace_filtered['value'] >= trace_filtered['global_p95']) |
    (trace_filtered['value'] <= trace_filtered['global_p5'])
)

# 4) Aggregate anomalies per (cmdb_id, trace_name)
trace_anomalies = trace_filtered[trace_filtered['is_anomaly']].copy()

trace_anomaly_summary = (
    trace_anomalies
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        earliest_anomaly_ts=('timestamp', 'min'),
        latest_anomaly_ts=('timestamp', 'max'),
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        global_p95=('global_p95', 'first'),
        global_p5=('global_p5', 'first'),
    )
)

# 6) Highlight the five candidate services first
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

candidates_df = trace_anomaly_summary[trace_anomaly_summary['cmdb_id'].isin(candidates)].sort_values('anomaly_count', ascending=False)
others_df = trace_anomaly_summary[~trace_anomaly_summary['cmdb_id'].isin(candidates)].sort_values('anomaly_count', ascending=False)

trace_anomaly_summary_top20 = pd.concat([candidates_df, others_df], ignore_index=True).head(20)
# Ensure columns order as requested
trace_anomaly_summary_top20 = trace_anomaly_summary_top20.loc[:, [
    'cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_ts', 'latest_anomaly_ts',
    'max_value', 'min_value', 'global_p95', 'global_p5'
]]

# 5B) Prepare sample anomaly detail rows limited to top 50 sorted by timestamp ascending,
# prioritizing candidate services first, then fill with others if needed
candidate_detail = trace_anomalies[trace_anomalies['cmdb_id'].isin(candidates)].copy().sort_values('timestamp', ascending=True)
other_detail = trace_anomalies[~trace_anomalies['cmdb_id'].isin(candidates)].copy().sort_values('timestamp', ascending=True)

# Take up to 50, preferring candidate rows
candidate_needed = 50
detail_frames = []
if not candidate_detail.empty:
    take_cand = candidate_detail.head(candidate_needed)
    detail_frames.append(take_cand)
    remaining = candidate_needed - len(take_cand)
    if remaining > 0 and not other_detail.empty:
        detail_frames.append(other_detail.head(remaining))
else:
    # No candidate anomalies: just take top 50 by timestamp from others
    detail_frames.append(other_detail.head(candidate_needed))

trace_anomaly_detail_top50 = pd.concat(detail_frames, ignore_index=True).sort_values('timestamp', ascending=True).loc[:, [
    'timestamp', 'cmdb_id', 'trace_name', 'value', 'global_p95', 'global_p5'
]].head(50).reset_index(drop=True)

# Store full filtered and anomaly tables in kernel variables for next steps:
# trace_filtered, trace_anomalies

# Display compact outputs
trace_anomaly_summary_top20, trace_anomaly_detail_top50

```
Out[4]:
```
```python
# Build a concise plain-English summary of the trace anomaly results using existing kernel variables
try:
    trace_anomalies
    trace_anomaly_summary_top20
    trace_anomaly_detail_top50
    candidates
except NameError:
    raise RuntimeError("Required variables (trace_anomalies, trace_anomaly_summary_top20, trace_anomaly_detail_top50, candidates) not found in kernel.")

total_trace_anomalies = int(trace_anomalies.shape[0])
earliest_anom = trace_anomalies['timestamp'].min()
latest_anom = trace_anomalies['timestamp'].max()

# Candidate-focused aggregates
candidate_mask = trace_anomalies['cmdb_id'].isin(candidates)
candidate_trace_anomalies = int(trace_anomalies[candidate_mask].shape[0])
candidate_distinct_groups = int(
    trace_anomalies[candidate_mask].drop_duplicates(subset=['cmdb_id', 'trace_name']).shape[0]
)

# Top 5 candidate trace groups by anomaly count (from the returned top20 summary)
top_candidates = (
    trace_anomaly_summary_top20[trace_anomaly_summary_top20['cmdb_id'].isin(candidates)]
    .sort_values('anomaly_count', ascending=False)
    .head(5)
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count']]
)
top_candidates_list = [f"{r.cmdb_id} / {r.trace_name} (count={int(r.anomaly_count)})"
                       for _, r in top_candidates.iterrows()]

summary = (
    "Trace anomaly analysis summary (plain English):\n"
    f"- Time range of detected trace anomalies: {earliest_anom.isoformat()} → {latest_anom.isoformat()} (UTC).\n"
    f"- Total trace anomaly points detected (all services) in the incident window: {total_trace_anomalies}.\n"
    f"- For the five candidate services ({', '.join(candidates)}):\n"
    f"  • Total anomaly points: {candidate_trace_anomalies}.\n"
    f"  • Distinct (cmdb_id, trace_name) groups affected: {candidate_distinct_groups}.\n\n"
    "Key observations (candidates prioritized):\n"
    f"- Top candidate trace groups by anomaly count:\n    " + "\n    ".join(top_candidates_list) + "\n\n"
    "- Many flagged anomalies are trace.error_rate series with global P95 and P5 equal to 0.0; when the observed values are 0.0, they meet the anomaly condition (value >= P95 or <= P5), which results in every point being marked anomalous for those series.\n"
    "- There are also non-error_rate anomalies (e.g., some duration_mean series such as trace.to_ts-order-service.duration_mean) where observed durations exceed the global P95 and were marked anomalous.\n\n"
    "Stored variables for follow-up:\n"
    "- trace_filtered   (full trace rows filtered to the incident window)\n"
    "- trace_anomalies  (full anomaly rows within the window)\n"
    "- trace_anomaly_summary_top20 (aggregated summary with candidates prioritized)\n"
    "- trace_anomaly_detail_top50 (sample anomaly detail rows, prioritized by candidates)\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                 trace_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value  min_value  global_p95  global_p5
0     ts-auth-service                 trace.from_root.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
1     ts-auth-service      trace.from_ts-auth-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
2    ts-order-service    trace.from_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
3    ts-order-service  trace.from_ts-security-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
4    ts-order-service  trace.from_ts-preserve-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
5    ts-order-service     trace.from_ts-order-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
6    ts-order-service                 trace.from_root.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
7     ts-auth-service        trace.to_ts-auth-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
8   ts-travel-service      trace.from_ts-seat-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
9   ts-travel-service    trace.from_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
10  ts-travel-service       trace.to_ts-route-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
11  ts-travel-service       trace.to_ts-order-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
12  ts-travel-service       trace.to_ts-train-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
13  ts-travel-service      trace.to_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
14   ts-order-service      trace.from_ts-seat-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
15   ts-order-service       trace.to_ts-order-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
16   ts-route-service    trace.from_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
17   ts-route-service   trace.from_ts-travel2-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
18   ts-train-service     trace.from_ts-basic-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0
19   ts-route-service       trace.to_ts-route-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0         0.0        0.0,                    timestamp            cmdb_id                                   trace_name     value  global_p95  global_p5
0  2024-02-20 18:31:00+00:00    ts-auth-service                   trace.from_root.error_rate  0.000000    0.000000   0.000000
1  2024-02-20 18:31:00+00:00  ts-travel-service      trace.to_ts-order-service.duration_mean  0.010077    0.008547   0.005286
2  2024-02-20 18:31:00+00:00  ts-travel-service      trace.from_ts-travel-service.error_rate  0.000000    0.000000   0.000000
3  2024-02-20 18:31:00+00:00  ts-travel-service        trace.from_ts-seat-service.error_rate  0.000000    0.000000   0.000000
4  2024-02-20 18:31:00+00:00  ts-travel-service    trace.from_ts-preserve-service.error_rate  0.000000    0.000000   0.000000
..                       ...                ...                                          ...       ...         ...        ...
45 2024-02-20 18:31:00+00:00  ts-travel-service         trace.to_ts-train-service.error_rate  0.000000    0.000000   0.000000
46 2024-02-20 18:31:00+00:00  ts-travel-service    trace.to_ts-ticketinfo-service.error_rate  0.000000    0.000000   0.000000
47 2024-02-20 18:31:00+00:00  ts-travel-service          trace.to_ts-seat-service.error_rate  0.000000    0.000000   0.000000
48 2024-02-20 18:31:00+00:00   ts-route-service         trace.to_ts-route-service.error_rate  0.000000    0.000000   0.000000
49 2024-02-20 18:31:00+00:00   ts-route-service  trace.from_ts-travel2-service.duration_mean  0.003547    0.003504   0.003239

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing df_log if present; otherwise load and parse timestamps as UTC
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed as UTC
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P5 for each (cmdb_id, log_name) using the full log series
log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(global_p95=lambda s: s.quantile(0.95), global_p5=lambda s: s.quantile(0.05))
    .reset_index()
)

# 2) Filter log rows to the incident window [2024-02-20 18:28:18 UTC, 2024-02-20 18:58:18 UTC]
start_ts = pd.Timestamp("2024-02-20 18:28:18", tz='UTC')
end_ts   = pd.Timestamp("2024-02-20 18:58:18", tz='UTC')
log_filtered = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# 3) Join global thresholds and mark anomalies: value >= global_p95 OR value <= global_p5
log_filtered = log_filtered.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')
log_filtered['is_anomaly'] = (
    (log_filtered['value'] >= log_filtered['global_p95']) |
    (log_filtered['value'] <= log_filtered['global_p5'])
)

# 4) Aggregate anomalies per (cmdb_id, log_name)
log_anomalies = log_filtered[log_filtered['is_anomaly']].copy()

log_anomaly_summary = (
    log_anomalies
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        anomaly_count=('is_anomaly', 'size'),
        earliest_anomaly_ts=('timestamp', 'min'),
        latest_anomaly_ts=('timestamp', 'max'),
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        global_p95=('global_p95', 'first'),
        global_p5=('global_p5', 'first'),
    )
)

# 6) Highlight the five candidate services first
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

candidates_df = log_anomaly_summary[log_anomaly_summary['cmdb_id'].isin(candidates)].sort_values('anomaly_count', ascending=False)
others_df = log_anomaly_summary[~log_anomaly_summary['cmdb_id'].isin(candidates)].sort_values('anomaly_count', ascending=False)

log_anomaly_summary_top20 = pd.concat([candidates_df, others_df], ignore_index=True).head(20)
log_anomaly_summary_top20 = log_anomaly_summary_top20.loc[:, [
    'cmdb_id', 'log_name', 'anomaly_count', 'earliest_anomaly_ts', 'latest_anomaly_ts',
    'max_value', 'min_value', 'global_p95', 'global_p5'
]]

# 5B) Prepare anomaly detail rows limited to top 100 sorted by timestamp ascending,
# prioritizing candidate services first, then fill with others if needed
candidate_detail = log_anomalies[log_anomalies['cmdb_id'].isin(candidates)].copy().sort_values('timestamp', ascending=True)
other_detail = log_anomalies[~log_anomalies['cmdb_id'].isin(candidates)].copy().sort_values('timestamp', ascending=True)

detail_limit = 100
detail_frames = []
if not candidate_detail.empty:
    take_cand = candidate_detail.head(detail_limit)
    detail_frames.append(take_cand)
    remaining = detail_limit - len(take_cand)
    if remaining > 0 and not other_detail.empty:
        detail_frames.append(other_detail.head(remaining))
else:
    detail_frames.append(other_detail.head(detail_limit))

log_anomaly_detail_top100 = (
    pd.concat(detail_frames, ignore_index=True)
      .sort_values('timestamp', ascending=True)
      .loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value', 'global_p95', 'global_p5']]
      .head(100)
      .reset_index(drop=True)
)

# Keep full filtered and anomaly tables in kernel variables for later correlation steps:
# log_filtered, log_anomalies

# Return compact outputs
log_anomaly_summary_top20, log_anomaly_detail_top100

```
Out[5]:
```
```python
# Summarize log anomaly results in plain English using existing kernel variables
try:
    log_anomalies
    log_anomaly_summary_top20
    log_anomaly_detail_top100
    candidates
except NameError:
    raise RuntimeError("Required variables (log_anomalies, log_anomaly_summary_top20, log_anomaly_detail_top100, candidates) not found.")

import pandas as pd

total_log_anomalies = int(log_anomalies.shape[0])
candidate_mask = log_anomalies['cmdb_id'].isin(candidates)
candidate_log_anomalies = int(log_anomalies[candidate_mask].shape[0])
candidate_distinct_groups = int(
    log_anomalies[candidate_mask].drop_duplicates(subset=['cmdb_id', 'log_name']).shape[0]
)

# Top candidate log groups by anomaly count from the summary (limited results)
top_candidate_groups = (
    log_anomaly_summary_top20[log_anomaly_summary_top20['cmdb_id'].isin(candidates)]
    .sort_values('anomaly_count', ascending=False)
    .head(10)
    .loc[:, ['cmdb_id', 'log_name', 'anomaly_count']]
)
top_list = [f"{r.cmdb_id} / {r.log_name} (count={int(r.anomaly_count)})" for _, r in top_candidate_groups.iterrows()]

summary = (
    "Log anomaly analysis summary (plain English):\n"
    f"- Total log anomaly points detected in the incident window: {total_log_anomalies}.\n"
    f"- For the five candidate services ({', '.join(candidates)}):\n"
    f"  • Total anomaly points: {candidate_log_anomalies}.\n"
    f"  • Distinct (cmdb_id, log_name) groups affected: {candidate_distinct_groups}.\n\n"
    "Key observations (candidates prioritized):\n"
    "- The candidate services show repeated anomalies for log.error_count (every point flagged) — global P95 and P5 for these error_count series are 0.0, so all zero-valued points meet the anomaly condition.\n"
    "- Candidate services also show anomalous log.row_count entries (a smaller number of points, e.g., 4) where row_count exceeded the computed P95 for that series.\n"
    f"- Top candidate log groups by anomaly count:\n    " + ("\n    ".join(top_list) if top_list else "None") + "\n\n"
    "Notes:\n"
    "- Full filtered log table is stored in: log_filtered\n"
    "- Full log anomaly table is stored in: log_anomalies\n"
    "- Sample anomaly detail rows are stored in: log_anomaly_detail_top100\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id         log_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value  min_value  global_p95  global_p5
0               ts-auth-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
1              ts-order-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
2             ts-travel-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
3              ts-route-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
4              ts-order-service    log.row_count              4 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00        553        154       530.8      309.0
5               ts-auth-service    log.row_count              4 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        402        120       391.2      269.6
6              ts-route-service    log.row_count              4 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00       1287        370      1233.8      841.0
7             ts-travel-service    log.row_count              4 2024-02-20 18:32:00+00:00 2024-02-20 18:55:00+00:00       2434        717      2354.2     1417.0
8   ts-admin-basic-info-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
9       ts-admin-travel-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
10             ts-basic-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
11         ts-assurance-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
12          ts-security-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
13       ts-order-other-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
14          ts-food-map-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
15              ts-ui-dashboard  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
16           ts-travel2-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
17              ts-seat-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
18          ts-preserve-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0
19             ts-price-service  log.error_count             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00          0          0         0.0        0.0,                    timestamp            cmdb_id         log_name  value  global_p95  global_p5
0  2024-02-20 18:31:00+00:00    ts-auth-service  log.error_count      0         0.0        0.0
1  2024-02-20 18:31:00+00:00    ts-auth-service    log.row_count    252       391.2      269.6
2  2024-02-20 18:31:00+00:00   ts-order-service  log.error_count      0         0.0        0.0
3  2024-02-20 18:31:00+00:00   ts-route-service  log.error_count      0         0.0        0.0
4  2024-02-20 18:31:00+00:00   ts-route-service    log.row_count    806      1233.8      841.0
..                       ...                ...              ...    ...         ...        ...
95 2024-02-20 18:51:00+00:00  ts-travel-service  log.error_count      0         0.0        0.0
96 2024-02-20 18:52:00+00:00   ts-route-service  log.error_count      0         0.0        0.0
97 2024-02-20 18:52:00+00:00    ts-auth-service  log.error_count      0         0.0        0.0
98 2024-02-20 18:52:00+00:00  ts-travel-service  log.error_count      0         0.0        0.0
99 2024-02-20 18:52:00+00:00   ts-order-service  log.error_count      0         0.0        0.0

[100 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd

# Ensure trace_anomaly_summary exists
try:
    trace_anomaly_summary
except NameError:
    raise RuntimeError("trace_anomaly_summary not found in kernel. Run trace anomaly computation first.")

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Build regex for candidate names
candidate_regex = "|".join([c for c in candidates])

# Filter for inter-service calls ('to_ts-' or 'from_ts-') that mention any candidate service
mask_call = trace_anomaly_summary['trace_name'].str.contains(r'(to_ts-|from_ts-)', regex=True, na=False)
mask_candidate = trace_anomaly_summary['trace_name'].str.contains(candidate_regex, regex=True, na=False)

trace_candidate_call_anomalies = (
    trace_anomaly_summary[mask_call & mask_candidate]
    .loc[:, ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_ts', 'latest_anomaly_ts',
             'max_value', 'min_value', 'global_p95', 'global_p5']]
    .sort_values('earliest_anomaly_ts', ascending=True)
    .head(50)
    .reset_index(drop=True)
)

# Per-candidate aggregation: earliest anomaly timestamp and total anomaly points
candidate_trace_aggregates = (
    trace_candidate_call_anomalies
    .groupby('cmdb_id', as_index=False)
    .agg(
        earliest_trace_anomaly_ts=('earliest_anomaly_ts', 'min'),
        total_trace_anomaly_points=('anomaly_count', 'sum')
    )
    .reindex(columns=['cmdb_id', 'earliest_trace_anomaly_ts', 'total_trace_anomaly_points'])
    .sort_values('earliest_trace_anomaly_ts', ascending=True)
    .reset_index(drop=True)
)

# Store trace_candidate_call_anomalies in kernel variable as requested (already assigned)
trace_candidate_call_anomalies, candidate_trace_aggregates

```
Out[6]:
```
```python
# Summarize the filtered trace inter-service call anomalies for candidate services
try:
    trace_candidate_call_anomalies
    candidate_trace_aggregates
except NameError:
    raise RuntimeError("Required variables trace_candidate_call_anomalies or candidate_trace_aggregates not found in kernel.")

# Basic counts and time bounds
num_rows = int(trace_candidate_call_anomalies.shape[0])
earliest_overall = trace_candidate_call_anomalies['earliest_anomaly_ts'].min()
latest_overall = trace_candidate_call_anomalies['latest_anomaly_ts'].max()

# Build per-candidate summary lines
per_candidate_lines = []
for _, row in candidate_trace_aggregates.iterrows():
    cm = row['cmdb_id']
    earliest = pd.Timestamp(row['earliest_trace_anomaly_ts']).isoformat()
    total_points = int(row['total_trace_anomaly_points'])
    per_candidate_lines.append(f"- {cm}: earliest anomaly at {earliest}, total trace-anomaly points = {total_points}")

# Identify a few representative trace_name patterns from the filtered set (up to 6)
rep_trace_names = (trace_candidate_call_anomalies['trace_name']
                   .drop_duplicates()
                   .head(6)
                   .tolist())

rep_lines = "\n".join([f"  • {n}" for n in rep_trace_names])

summary = (
    "Trace inter-service call anomaly summary (plain English):\n"
    f"- Rows matching candidate services & inter-service calls: {num_rows} rows (limited view stored in variable trace_candidate_call_anomalies).\n"
    f"- Time range of these anomalies: {earliest_overall.isoformat()} → {latest_overall.isoformat()} (UTC).\n\n"
    "Per-candidate aggregates:\n"
    + "\n".join(per_candidate_lines) + "\n\n"
    "Representative affected inter-service trace names (examples):\n"
    f"{rep_lines}\n\n"
    "Key observation:\n"
    "- Many flagged inter-service entries are error_rate series with global P95/P5 equal to 0.0, which causes all-zero series to meet the anomaly condition (so those series appear fully anomalous).\n"
    "- There are also non-error_rate call anomalies (e.g., row_count or duration_mean) where observed values exceeded global P95; these are likely more meaningful performance anomalies for inter-service calls.\n\n"
    "Stored for next steps:\n"
    "- trace_candidate_call_anomalies: filtered inter-service call anomaly rows\n"
    "- candidate_trace_aggregates: per-candidate earliest anomaly ts and total trace-anomaly points\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                               trace_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts   max_value  min_value  global_p95   global_p5
0          ts-travel2-service      trace.to_ts-train-service.row_count              4 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  348.000000  95.000000  338.600000  222.800000
1         ts-preserve-service    trace.to_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    0.000000   0.000000    0.000000    0.000000
2          ts-station-service   trace.from_ts-order-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    0.000000   0.000000    0.000000    0.000000
3       ts-ticketinfo-service  trace.from_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    0.000000   0.000000    0.000000    0.000000
4         ts-preserve-service     trace.to_ts-order-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    0.000000   0.000000    0.000000    0.000000
..                        ...                                      ...            ...                       ...                       ...         ...        ...         ...         ...
45          ts-travel-service    trace.to_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    0.000000   0.000000    0.000000    0.000000
46                       root     trace.to_ts-travel-service.row_count              4 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   49.000000   9.000000   35.600000   18.200000
47  ts-inside-payment-service     trace.to_ts-order-service.error_rate             24 2024-02-20 18:31:00+00:00 2024-02-20 18:54:00+00:00    0.000000   0.000000    0.000000    0.000000
48                       root    trace.to_ts-travel-service.error_rate             25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    0.000000   0.000000    0.000000    0.000000
49         ts-travel2-service  trace.to_ts-route-service.duration_mean              4 2024-02-20 18:31:00+00:00 2024-02-20 18:53:00+00:00    0.003547   0.003194    0.003504    0.003239

[50 rows x 9 columns],                       cmdb_id earliest_trace_anomaly_ts  total_trace_anomaly_points
0                        root 2024-02-20 18:31:00+00:00                          29
1     ts-admin-travel-service 2024-02-20 18:31:00+00:00                          29
2             ts-auth-service 2024-02-20 18:31:00+00:00                          29
3   ts-inside-payment-service 2024-02-20 18:31:00+00:00                          24
4            ts-order-service 2024-02-20 18:31:00+00:00                          83
5         ts-preserve-service 2024-02-20 18:31:00+00:00                          58
6            ts-route-service 2024-02-20 18:31:00+00:00                         103
7             ts-seat-service 2024-02-20 18:31:00+00:00                          58
8         ts-security-service 2024-02-20 18:31:00+00:00                          33
9          ts-station-service 2024-02-20 18:31:00+00:00                          33
10      ts-ticketinfo-service 2024-02-20 18:31:00+00:00                          25
11           ts-train-service 2024-02-20 18:31:00+00:00                          91
12          ts-travel-service 2024-02-20 18:31:00+00:00                          75
13         ts-travel2-service 2024-02-20 18:31:00+00:00                          33)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import re
import pandas as pd

# Reuse existing variable trace_candidate_call_anomalies
try:
    trace_candidate_call_anomalies
except NameError:
    raise RuntimeError("trace_candidate_call_anomalies not found in kernel. Run prior steps first.")

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Work on rows where trace_name contains 'to_ts-'
df_calls = trace_candidate_call_anomalies[trace_candidate_call_anomalies['trace_name'].str.contains('to_ts-', na=False)].copy()

# Parse target service and metric type from trace_name
def parse_target_metric(tname):
    # match 'to_ts-<service>[.]<metric>' or 'to_ts-<service>.<rest>'
    m = re.search(r"to_ts-([a-z0-9-]+)(?:\.(.+))?$", tname)
    if not m:
        return None, None
    svc_part = m.group(1)
    metric_part = m.group(2) or ""
    # ensure target service has 'ts-' prefix as in candidates
    target = svc_part if svc_part.startswith('ts-') else f"ts-{svc_part}"
    # metric type could include dots; take the first token (e.g., error_rate, duration_mean, row_count, duration_p95)
    metric_type = metric_part.split('.')[0] if metric_part else ""
    return target, metric_type

parsed = df_calls['trace_name'].apply(lambda x: pd.Series(parse_target_metric(x), index=['target_service', 'metric_type']))
df_calls = pd.concat([df_calls.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

# Keep only edges where target_service is one of the candidates
df_edges = df_calls[df_calls['target_service'].isin(candidates)].copy()

# If no edges, create empty results
if df_edges.empty:
    trace_candidate_edges = pd.DataFrame(columns=[
        'source_cmdb_id', 'target_service', 'trace_metric_types', 'total_anomaly_count',
        'earliest_anomaly_ts', 'latest_anomaly_ts', 'max_value', 'min_value'
    ])
    candidate_target_summary = pd.DataFrame({
        'target_service': candidates,
        'earliest_incoming_anomaly_ts': [pd.NaT]*len(candidates),
        'distinct_source_count': [0]*len(candidates)
    })
else:
    # Normalize source column name
    df_edges = df_edges.rename(columns={'cmdb_id': 'source_cmdb_id'})

    # Aggregate by edge (source -> target)
    agg = (
        df_edges
        .groupby(['source_cmdb_id', 'target_service'], as_index=False)
        .agg(
            earliest_anomaly_ts=('earliest_anomaly_ts', 'min'),
            latest_anomaly_ts=('latest_anomaly_ts', 'max'),
            total_anomaly_count=('anomaly_count', 'sum'),
            max_value=('max_value', 'max'),
            min_value=('min_value', 'min'),
            trace_metric_types=('metric_type', lambda s: ';'.join(sorted(pd.Series(s.dropna().unique()))))
        )
    )

    # Reorder columns as requested and sort by earliest_anomaly_ts ascending, limit to top 20
    trace_candidate_edges = agg.loc[:, [
        'source_cmdb_id', 'target_service', 'trace_metric_types', 'total_anomaly_count',
        'earliest_anomaly_ts', 'latest_anomaly_ts', 'max_value', 'min_value'
    ]].sort_values('earliest_anomaly_ts', ascending=True).reset_index(drop=True)

    # Per-target summary: earliest incoming anomaly ts and number of distinct source_cmdb_id
    per_target = (
        trace_candidate_edges
        .groupby('target_service', as_index=False)
        .agg(
            earliest_incoming_anomaly_ts=('earliest_anomaly_ts', 'min'),
            distinct_source_count=('source_cmdb_id', 'nunique')
        )
    )

    # Ensure all five candidates are present in summary (add missing with NaT/0)
    missing = [c for c in candidates if c not in per_target['target_service'].values]
    if missing:
        add_df = pd.DataFrame({
            'target_service': missing,
            'earliest_incoming_anomaly_ts': [pd.NaT]*len(missing),
            'distinct_source_count': [0]*len(missing)
        })
        per_target = pd.concat([per_target, add_df], ignore_index=True)

    candidate_target_summary = per_target.loc[:, ['target_service', 'earliest_incoming_anomaly_ts', 'distinct_source_count']].set_index('target_service').loc[candidates].reset_index()

# Store aggregated edge table in kernel variable for next steps
trace_candidate_edges = trace_candidate_edges.head(20)  # keep compact in memory view

# Return compact results
trace_candidate_edges, candidate_target_summary

```
Out[7]:
```
```python
# Build a plain-English summary from trace_candidate_edges and candidate_target_summary
try:
    trace_candidate_edges
    candidate_target_summary
except NameError:
    raise RuntimeError("Required variables not found in kernel.")

# Compact counts and samples
num_edges = int(trace_candidate_edges.shape[0])
edges_sample = trace_candidate_edges.head(5)

edge_lines = []
for _, r in edges_sample.iterrows():
    line = (f"{r['source_cmdb_id']} -> {r['target_service']}: metrics={r['trace_metric_types']}, "
            f"count={int(r['total_anomaly_count'])}, earliest={pd.Timestamp(r['earliest_anomaly_ts']).isoformat()}, "
            f"latest={pd.Timestamp(r['latest_anomaly_ts']).isoformat()}, max={r['max_value']}, min={r['min_value']}")
    edge_lines.append(line)

# Per-target summary
target_lines = []
for _, r in candidate_target_summary.iterrows():
    ts = r['earliest_incoming_anomaly_ts']
    ts_str = pd.Timestamp(ts).isoformat() if pd.notna(ts) else "none"
    target_lines.append(f"{r['target_service']}: earliest incoming anomaly = {ts_str}, distinct source count = {int(r['distinct_source_count'])}")

summary = (
    "Trace candidate-to-candidate call anomaly summary (plain English):\n"
    f"- Extracted candidate incoming-call edges (target is one of 5 candidates): {num_edges} edges (top 20 shown).\n\n"
    "Top example edges (by earliest anomaly timestamp):\n"
    + "\n".join(f"  {i+1}. {l}" for i, l in enumerate(edge_lines)) + "\n\n"
    "Per-target summary (earliest incoming anomaly and number of distinct sources):\n"
    + "\n".join(f"  {l}" for l in target_lines) + "\n\n"
    "Key observations:\n"
    "- Several sources (e.g., root, ts-admin-travel-service, ts-preserve-service, ts-travel-service, ts-travel2-service) show anomalous incoming calls to candidate services.\n"
    "- Many edges include 'error_rate' series with global P95/P5 = 0.0 (resulting in all-zero series marked anomalous). More actionable edges also include duration_mean, duration_p95, or row_count anomalies.\n"
    "- Among the five candidate targets, ts-order-service, ts-route-service, ts-train-service, and ts-travel-service each had multiple distinct sources with anomalous incoming calls; ts-auth-service had none in this filtered set.\n\n"
    "Stored variable for next steps: trace_candidate_edges (aggregated edge table)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               source_cmdb_id     target_service                               trace_metric_types  total_anomaly_count       earliest_anomaly_ts         latest_anomaly_ts    max_value  min_value
0                        root  ts-travel-service                             error_rate;row_count                   29 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    49.000000   0.000000
1     ts-admin-travel-service  ts-travel-service                             error_rate;row_count                   29 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00    12.000000   0.000000
2   ts-inside-payment-service   ts-order-service                                       error_rate                   24 2024-02-20 18:31:00+00:00 2024-02-20 18:54:00+00:00     0.000000   0.000000
3            ts-order-service   ts-order-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.000000   0.000000
4         ts-preserve-service   ts-order-service            duration_mean;duration_p95;error_rate                   33 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.079473   0.000000
5         ts-preserve-service  ts-travel-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.000000   0.000000
6            ts-route-service   ts-route-service  duration_mean;duration_p95;error_rate;row_count                   37 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  2574.000000   0.000000
7             ts-seat-service   ts-order-service            duration_mean;duration_p95;error_rate                   33 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.045354   0.000000
8         ts-security-service   ts-order-service            duration_mean;duration_p95;error_rate                   33 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.082026   0.000000
9            ts-train-service   ts-train-service                duration_p95;error_rate;row_count                   33 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00  3180.000000   0.000000
10          ts-travel-service   ts-train-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.000000   0.000000
11          ts-travel-service  ts-travel-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     0.000000   0.000000
12         ts-travel2-service   ts-route-service                                    duration_mean                    4 2024-02-20 18:31:00+00:00 2024-02-20 18:53:00+00:00     0.003547   0.003194
13         ts-travel2-service   ts-train-service                             error_rate;row_count                   29 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00   348.000000   0.000000,       target_service earliest_incoming_anomaly_ts  distinct_source_count
0    ts-auth-service                          NaT                      0
1   ts-order-service    2024-02-20 18:31:00+00:00                      5
2   ts-route-service    2024-02-20 18:31:00+00:00                      2
3   ts-train-service    2024-02-20 18:31:00+00:00                      3
4  ts-travel-service    2024-02-20 18:31:00+00:00                      4)```
```

In [ ]:
# In[8]:

import pandas as pd

# Reuse existing kernel variables: metric_anomalies_full, trace_candidate_edges, candidate_target_summary, log_anomalies, candidate_anomaly_summary
try:
    metric_anomalies_full
    trace_candidate_edges
    candidate_target_summary
    log_anomalies
    candidate_anomaly_summary
except NameError as e:
    raise RuntimeError("One or more required variables are missing from kernel: " + str(e))

candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1-2) Metric anomalies: earliest timestamp and total points per candidate
metric_grp = (
    metric_anomalies_full
    .groupby('cmdb_id', observed=True)['timestamp']
    .agg(earliest_metric_anomaly_ts='min', total_metric_anomaly_points='count')
    .reset_index()
)

# 3) Log anomalies: earliest timestamp and total points per candidate
log_grp = (
    log_anomalies
    .groupby('cmdb_id', observed=True)['timestamp']
    .agg(earliest_log_anomaly_ts='min', total_log_anomaly_points='count')
    .reset_index()
)

# 4) Incoming trace anomalies: sum of total_anomaly_count per target and earliest incoming ts per candidate
# trace_candidate_edges has per-edge total_anomaly_count and earliest_anomaly_ts
if trace_candidate_edges.shape[0] > 0:
    trace_incoming_agg = (
        trace_candidate_edges
        .groupby('target_service', observed=True)
        .agg(
            earliest_incoming_trace_anomaly_ts=('earliest_anomaly_ts', 'min'),
            total_incoming_trace_anomaly_points=('total_anomaly_count', 'sum')
        )
        .reset_index()
    )
else:
    trace_incoming_agg = pd.DataFrame(columns=['target_service', 'earliest_incoming_trace_anomaly_ts', 'total_incoming_trace_anomaly_points'])

# Ensure all candidates present in each aggregation (fill missing with NaT/0)
def ensure_candidates_df(df, key_col, ts_col, count_col):
    df = df.copy()
    missing = [c for c in candidates if c not in df[key_col].values]
    if missing:
        add = pd.DataFrame({
            key_col: missing,
            ts_col: [pd.NaT]*len(missing),
            count_col: [0]*len(missing)
        })
        df = pd.concat([df, add], ignore_index=True)
    return df

metric_grp = ensure_candidates_df(metric_grp, 'cmdb_id', 'earliest_metric_anomaly_ts', 'total_metric_anomaly_points')
log_grp = ensure_candidates_df(log_grp, 'cmdb_id', 'earliest_log_anomaly_ts', 'total_log_anomaly_points')
trace_incoming_agg = ensure_candidates_df(trace_incoming_agg, 'target_service', 'earliest_incoming_trace_anomaly_ts', 'total_incoming_trace_anomaly_points')

# 5) Top metric KPI by anomaly_count per candidate from candidate_anomaly_summary
# candidate_anomaly_summary contains per (cmdb_id, kpi_name) anomaly_count
top_metric = (
    candidate_anomaly_summary
    .sort_values(['cmdb_id', 'anomaly_count'], ascending=[True, False])
    .groupby('cmdb_id', observed=True, as_index=False)
    .first()[['cmdb_id', 'kpi_name', 'anomaly_count']]
    .rename(columns={'kpi_name': 'top_metric_kpi', 'anomaly_count': 'top_metric_kpi_anomaly_count'})
)

# Ensure top_metric covers all candidates
top_metric = ensure_candidates_df(top_metric, 'cmdb_id', 'top_metric_kpi', 'top_metric_kpi_anomaly_count')
# Fill missing text/zeros
top_metric['top_metric_kpi'] = top_metric['top_metric_kpi'].fillna('')
top_metric['top_metric_kpi_anomaly_count'] = top_metric['top_metric_kpi_anomaly_count'].fillna(0).astype(int)

# Merge all pieces into final table (one row per candidate, preserve candidate order)
df_final = pd.DataFrame({'cmdb_id': candidates})

# Left-join metric, log, trace incoming, and top_metric
df_final = (
    df_final
    .merge(metric_grp, how='left', left_on='cmdb_id', right_on='cmdb_id')
    .merge(top_metric, how='left', left_on='cmdb_id', right_on='cmdb_id')
    .merge(log_grp, how='left', left_on='cmdb_id', right_on='cmdb_id')
    .merge(trace_incoming_agg.rename(columns={'target_service': 'cmdb_id'}), how='left', left_on='cmdb_id', right_on='cmdb_id')
)

# Replace NaN counts with 0 and keep timestamps as-is (NaT where missing)
df_final['total_metric_anomaly_points'] = df_final['total_metric_anomaly_points'].fillna(0).astype(int)
df_final['total_log_anomaly_points'] = df_final['total_log_anomaly_points'].fillna(0).astype(int)
df_final['total_incoming_trace_anomaly_points'] = df_final['total_incoming_trace_anomaly_points'].fillna(0).astype(int)

# Reorder columns per request
candidate_final_timeline = df_final.loc[:, [
    'cmdb_id',
    'earliest_metric_anomaly_ts',
    'total_metric_anomaly_points',
    'top_metric_kpi',
    'top_metric_kpi_anomaly_count',
    'earliest_log_anomaly_ts',
    'total_log_anomaly_points',
    'earliest_incoming_trace_anomaly_ts',
    'total_incoming_trace_anomaly_points'
]]

# Ensure compact display: one row per candidate (5 rows)
candidate_final_timeline

```
Out[8]:
```
Summary per candidate (plain English):

- ts-auth-service
  - Earliest metric anomaly: 2024-02-20T18:31:00Z
  - Total metric anomaly points: 28
  - Top metric KPI with most anomalies: cpu (4 anomalies)
  - Earliest log anomaly: 2024-02-20T18:31:00Z
  - Total log anomaly points: 29
  - Earliest incoming trace anomaly: none detected
  - Total incoming trace anomaly points: 0

- ts-order-service
  - Earliest metric anomaly: 2024-02-20T18:31:00Z
  - Total metric anomaly points: 29
  - Top metric KPI with most anomalies: socket (5 anomalies)
  - Earliest log anomaly: 2024-02-20T18:31:00Z
  - Total log anomaly points: 29
  - Earliest incoming trace anomaly: 2024-02-20T18:31:00Z
  - Total incoming trace anomaly points: 148

- ts-route-service
  - Earliest metric anomaly: 2024-02-20T18:31:00Z
  - Total metric anomaly points: 28
  - Top metric KPI with most anomalies: cpu (4 anomalies)
  - Earliest log anomaly: 2024-02-20T18:31:00Z
  - Total log anomaly points: 29
  - Earliest incoming trace anomaly: 2024-02-20T18:31:00Z
  - Total incoming trace anomaly points: 41

- ts-train-service
  - Earliest metric anomaly: 2024-02-20T18:31:00Z
  - Total metric anomaly points: 29
  - Top metric KPI with most anomalies: socket (5 anomalies)
  - Earliest log anomaly: none detected
  - Total log anomaly points: 0
  - Earliest incoming trace anomaly: 2024-02-20T18:31:00Z
  - Total incoming trace anomaly points: 87

- ts-travel-service
  - Earliest metric anomaly: 2024-02-20T18:31:00Z
  - Total metric anomaly points: 34
  - Top metric KPI with most anomalies: error (6 anomalies)
  - Earliest log anomaly: 2024-02-20T18:31:00Z
  - Total log anomaly points: 29
  - Earliest incoming trace anomaly: 2024-02-20T18:31:00Z
  - Total incoming trace anomaly points: 108

Overall: metric, log, and trace anomalies for these services begin around 2024-02-20T18:31:00Z. ts-order-service, ts-train-service, ts-route-service, and ts-travel-service show substantial incoming trace anomaly counts (many anomalous incoming calls); ts-auth-service shows metric and log anomalies but no anomalous incoming calls in the filtered trace edges.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_metric_anomaly_ts  total_metric_anomaly_points top_metric_kpi  top_metric_kpi_anomaly_count    earliest_log_anomaly_ts  total_log_anomaly_points earliest_incoming_trace_anomaly_ts  total_incoming_trace_anomaly_points
0    ts-auth-service  2024-02-20 18:31:00+00:00                           28            cpu                             4  2024-02-20 18:31:00+00:00                        29                                NaT                                    0
1   ts-order-service  2024-02-20 18:31:00+00:00                           29         socket                             5  2024-02-20 18:31:00+00:00                        29          2024-02-20 18:31:00+00:00                                  148
2   ts-route-service  2024-02-20 18:31:00+00:00                           28            cpu                             4  2024-02-20 18:31:00+00:00                        29          2024-02-20 18:31:00+00:00                                   41
3   ts-train-service  2024-02-20 18:31:00+00:00                           29         socket                             5                        NaT                         0          2024-02-20 18:31:00+00:00                                   87
4  ts-travel-service  2024-02-20 18:31:00+00:00                           34          error                             6  2024-02-20 18:31:00+00:00                        29          2024-02-20 18:31:00+00:00                                  108```
```

In [ ]:
# In[9]:

import pandas as pd

# Reuse existing variables: trace_candidate_edges, candidates
try:
    trace_candidate_edges
    candidates
except NameError as e:
    raise RuntimeError("Required variables not found in kernel: " + str(e))

# 1) Filter edges where both source and target are among the five candidates
candidate_to_candidate_edges = trace_candidate_edges[
    trace_candidate_edges['source_cmdb_id'].isin(candidates) &
    trace_candidate_edges['target_service'].isin(candidates)
].copy()

# 2) Return up to 20 rows sorted by earliest_anomaly_ts ascending with requested columns
candidate_to_candidate_edges = candidate_to_candidate_edges.loc[:, [
    'source_cmdb_id', 'target_service', 'trace_metric_types', 'total_anomaly_count',
    'earliest_anomaly_ts', 'latest_anomaly_ts', 'max_value', 'min_value'
]].sort_values('earliest_anomaly_ts', ascending=True).head(20).reset_index(drop=True)

# 3) Per-target summary for the five candidates
# For each candidate target, determine if there are incoming edges from candidate sources
if not candidate_to_candidate_edges.empty:
    per_target = (
        candidate_to_candidate_edges
        .groupby('target_service', as_index=False)
        .agg(
            earliest_incoming_from_candidate_ts=('earliest_anomaly_ts', 'min'),
            number_of_distinct_candidate_sources=('source_cmdb_id', 'nunique')
        )
    )
    per_target['has_incoming_from_candidates'] = per_target['number_of_distinct_candidate_sources'] > 0
else:
    per_target = pd.DataFrame(columns=[
        'target_service', 'earliest_incoming_from_candidate_ts', 'number_of_distinct_candidate_sources',
        'has_incoming_from_candidates'
    ])

# Ensure all five candidates are present in the summary (fill missing)
missing = [c for c in candidates if c not in per_target['target_service'].values]
if missing:
    add = pd.DataFrame({
        'target_service': missing,
        'earliest_incoming_from_candidate_ts': [pd.NaT]*len(missing),
        'number_of_distinct_candidate_sources': [0]*len(missing),
        'has_incoming_from_candidates': [False]*len(missing)
    })
    per_target = pd.concat([per_target, add], ignore_index=True)

candidate_incoming_from_candidates = (
    per_target.loc[:, ['target_service', 'has_incoming_from_candidates',
                       'earliest_incoming_from_candidate_ts', 'number_of_distinct_candidate_sources']]
    .set_index('target_service')
    .loc[candidates]  # preserve candidate order
    .reset_index()
)

# Display compact outputs (variables)
candidate_to_candidate_edges, candidate_incoming_from_candidates

```
Out[9]:
```
```python
# Summarize candidate-to-candidate trace edges and per-target incoming info in plain English
try:
    candidate_to_candidate_edges
    candidate_incoming_from_candidates
except NameError:
    raise RuntimeError("Required variables not found in kernel.")

lines = []
lines.append("Candidate-to-candidate trace-edge summary (plain English):")

# Edge list summary (up to 20 rows present)
if candidate_to_candidate_edges.empty:
    lines.append("- No candidate-to-candidate anomalous call edges found.")
else:
    lines.append(f"- Found {len(candidate_to_candidate_edges)} candidate->candidate anomalous edges (sampled up to 20):")
    for _, r in candidate_to_candidate_edges.iterrows():
        src = r['source_cmdb_id']
        tgt = r['target_service']
        metrics = r['trace_metric_types']
        cnt = int(r['total_anomaly_count'])
        est = pd.Timestamp(r['earliest_anomaly_ts']).isoformat()
        let = pd.Timestamp(r['latest_anomaly_ts']).isoformat()
        maxv = r['max_value']
        minv = r['min_value']
        lines.append(f"  - {src} -> {tgt}: metrics={metrics}, total_anomaly_count={cnt}, earliest={est}, latest={let}, max={maxv}, min={minv}")

lines.append("")    
lines.append("Per-target incoming-from-candidates summary (five candidates):")
for _, r in candidate_incoming_from_candidates.iterrows():
    tgt = r['target_service']
    has = bool(r['has_incoming_from_candidates'])
    earliest = r['earliest_incoming_from_candidate_ts']
    earliest_str = pd.Timestamp(earliest).isoformat() if pd.notna(earliest) else "none"
    nsrc = int(r['number_of_distinct_candidate_sources'])
    lines.append(f"  - {tgt}: has_incoming_from_candidates={has}, earliest_incoming_from_candidate_ts={earliest_str}, distinct_candidate_sources={nsrc}")

lines.append("")    
lines.append("Key observations:")
lines.append("- Most candidate->candidate edges in the filtered set are self-call edges (service -> same service) flagged on trace.error_rate or other trace metrics.")
lines.append("- One cross-service anomalous edge among candidates: ts-travel-service -> ts-train-service (error_rate anomalies).")
lines.append("- ts-auth-service has no anomalous incoming calls from other candidates in this filtered view; the other four candidates have at least one incoming anomalous edge from candidate sources.")
lines.append("- Note: many flagged 'error_rate' series have global P95/P5 = 0.0, so zero-valued series were marked anomalous; duration/row_count anomalies are more indicative of performance issues.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      source_cmdb_id     target_service                               trace_metric_types  total_anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value  min_value
0   ts-order-service   ts-order-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0
1   ts-route-service   ts-route-service  duration_mean;duration_p95;error_rate;row_count                   37 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     2574.0        0.0
2   ts-train-service   ts-train-service                duration_p95;error_rate;row_count                   33 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00     3180.0        0.0
3  ts-travel-service   ts-train-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0
4  ts-travel-service  ts-travel-service                                       error_rate                   25 2024-02-20 18:31:00+00:00 2024-02-20 18:55:00+00:00        0.0        0.0,       target_service  has_incoming_from_candidates earliest_incoming_from_candidate_ts  number_of_distinct_candidate_sources
0    ts-auth-service                         False                                 NaT                                     0
1   ts-order-service                          True           2024-02-20 18:31:00+00:00                                     1
2   ts-route-service                          True           2024-02-20 18:31:00+00:00                                     1
3   ts-train-service                          True           2024-02-20 18:31:00+00:00                                     2
4  ts-travel-service                          True           2024-02-20 18:31:00+00:00                                     1)```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np

# Reuse existing variables from kernel:
# metric_anomalies_full, metric_filtered_full, candidate_anomaly_summary, metric_thresholds
try:
    metric_anomalies_full
    metric_filtered_full
    candidate_anomaly_summary
    metric_thresholds
except NameError as e:
    raise RuntimeError("Required variable missing from kernel: " + str(e))

candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Select top metric KPI by anomaly_count for each candidate
top_kpis = (
    candidate_anomaly_summary[candidate_anomaly_summary['cmdb_id'].isin(candidates)]
    .sort_values(['cmdb_id', 'anomaly_count'], ascending=[True, False])
    .groupby('cmdb_id', as_index=False)
    .first()[['cmdb_id', 'kpi_name', 'anomaly_count']]
)

# Ensure all candidates present; if missing, add rows with empty KPI
missing = [c for c in candidates if c not in top_kpis['cmdb_id'].values]
if missing:
    add = pd.DataFrame({'cmdb_id': missing, 'kpi_name': ['']*len(missing), 'anomaly_count': [0]*len(missing)})
    top_kpis = pd.concat([top_kpis, add], ignore_index=True)

# 2) Aggregate anomaly details from metric_anomalies_full for selected (cmdb_id, kpi_name)
sel = top_kpis[['cmdb_id', 'kpi_name']].copy()

# Prepare aggregated anomaly stats from metric_anomalies_full
anom_agg = (
    metric_anomalies_full
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count_filtered=('value', 'size'),
        earliest_anomaly_ts=('timestamp', 'min'),
        latest_anomaly_ts=('timestamp', 'max'),
        max_value=('value', 'max'),
        min_value=('value', 'min')
    )
)

# Minutes present for each KPI in incident-window filtered metric data
minutes_present_df = (
    metric_filtered_full
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(minutes_present=('value', 'count'))
)

# 3) Obtain global_p95 from metric_thresholds (precomputed)
# metric_thresholds columns: cmdb_id, kpi_name, global_p95, global_p5
thresholds = metric_thresholds[['cmdb_id', 'kpi_name', 'global_p95']].copy()

# Merge all info for selected KPIs
df = (
    sel
    .merge(anom_agg, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(minutes_present_df, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NA numeric fields with sensible defaults where appropriate
df['anomaly_count_filtered'] = df['anomaly_count_filtered'].fillna(0).astype(int)
df['max_value'] = df['max_value'].astype(float)
df['min_value'] = df['min_value'].astype(float)
df['minutes_present'] = df['minutes_present'].fillna(0).astype(int)
df['global_p95'] = df['global_p95'].astype(float)

# 4) Compute breach_ratio
df['breach_ratio'] = np.where(df['global_p95'] > 0,
                              (df['max_value'] - df['global_p95']) / df['global_p95'],
                              np.nan)

# 5) Compute persistence = anomaly_count / minutes_present
df['persistence'] = np.where(df['minutes_present'] > 0,
                             df['anomaly_count_filtered'] / df['minutes_present'],
                             np.nan)

# Prepare final columns and rename for clarity
candidate_severity_summary = df.rename(columns={
    'kpi_name': 'kpi_name',
    'anomaly_count_filtered': 'anomaly_count',
    'earliest_anomaly_ts': 'earliest_anomaly_ts',
    'latest_anomaly_ts': 'latest_anomaly_ts',
    'max_value': 'max_value',
    'global_p95': 'global_p95',
    'minutes_present': 'minutes_present',
    'persistence': 'persistence',
    'breach_ratio': 'breach_ratio'
})[[
    'cmdb_id',
    'kpi_name',
    'anomaly_count',
    'earliest_anomaly_ts',
    'latest_anomaly_ts',
    'max_value',
    'global_p95',
    'breach_ratio',
    'minutes_present',
    'persistence'
]]

# Sort by breach_ratio descending and limit to the five candidates (preserve candidate rows)
candidate_severity_summary = candidate_severity_summary.sort_values('breach_ratio', ascending=False).reset_index(drop=True)

# Store in kernel variable and display compact table
candidate_severity_summary

```
Out[12]:
```
```python
# Build a concise plain-English summary from candidate_severity_summary (already in kernel)
try:
    candidate_severity_summary
except NameError:
    raise RuntimeError("candidate_severity_summary not found in kernel.")

s = candidate_severity_summary

lines = ["Severity summary for five candidate services (plain English):\n"]

for _, r in s.iterrows():
    cm = r['cmdb_id']
    kpi = r['kpi_name']
    ac = int(r['anomaly_count'])
    est = pd.Timestamp(r['earliest_anomaly_ts']).isoformat() if pd.notna(r['earliest_anomaly_ts']) else "none"
    let = pd.Timestamp(r['latest_anomaly_ts']).isoformat() if pd.notna(r['latest_anomaly_ts']) else "none"
    maxv = r['max_value']
    p95 = r['global_p95']
    br = r['breach_ratio']
    mins = int(r['minutes_present'])
    pers = r['persistence']
    lines.append(
        f"- {cm}: top KPI = {kpi}. Anomalies: {ac} points from {est} to {let}. "
        f"Observed max = {maxv:.6g}, global P95 = {p95:.6g}, breach_ratio = "
        + (f"{br:.3f}" if pd.notna(br) else "NaN")
        + f". Minutes present = {mins}, persistence = "
        + (f"{pers:.2f}" if pd.notna(pers) else "NaN") + "."
    )

lines.append("\nInterpretation:")
lines.append("- ts-travel-service shows the most severe metric breach: error KPI with breach_ratio ~0.33 and persistence = 1.00 (all 6 minutes present were anomalous).")
lines.append("- ts-auth-service and ts-route-service have modest breaches on CPU (~3% and ~2% respectively) with low persistence (~16%).")
lines.append("- ts-order-service and ts-train-service show socket anomalies; ts-order has a small breach (~1%) with 20% persistence, ts-train's max equals its global P95 (breach_ratio 0) but still has 5 anomaly points (persistence 20%).")
lines.append("- Overall, ts-travel-service stands out as highest severity by breach_ratio and persistence; others show smaller breaches or intermittent anomalies.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value  global_p95  breach_ratio  minutes_present  persistence
0  ts-travel-service    error              6 2024-02-20 18:50:00+00:00 2024-02-20 18:55:00+00:00   0.023929    0.017946      0.333333                6         1.00
1    ts-auth-service      cpu              4 2024-02-20 18:34:00+00:00 2024-02-20 18:50:00+00:00  19.425825   18.852920      0.030388               25         0.16
2   ts-route-service      cpu              4 2024-02-20 18:32:00+00:00 2024-02-20 18:51:00+00:00   5.731560    5.617819      0.020246               25         0.16
3   ts-order-service   socket              5 2024-02-20 18:32:00+00:00 2024-02-20 18:53:00+00:00  14.783333   14.640000      0.009791               25         0.20
4   ts-train-service   socket              5 2024-02-20 18:34:00+00:00 2024-02-20 18:55:00+00:00  25.000000   25.000000      0.000000               25         0.20```
```